# AIPROD — Générer votre Première Vidéo sur Google Colab Pro

**GPU requis :** A100 40GB (Colab Pro)  
**Durée totale :** ~10 minutes (téléchargement modèle) + ~5 minutes (génération)  
**Coût :** Inclus dans Colab Pro (~10$/mois)  

---

### Ce que fait ce notebook

| Étape | Action | Durée |
|-------|--------|-------|
| 1 | Vérifier GPU A100 | 5 sec |
| 2 | Cloner le repo AIPROD | 30 sec |
| 3 | Installer les dépendances (diffusers + PyTorch) | 2-3 min |
| 4 | Authentification HuggingFace | 10 sec |
| 5 | **Générer une vidéo** (téléchargement modèle au 1er run) | 10-15 min |
| 6 | Télécharger le résultat | 10 sec |

> **Résultat :** Un fichier `.mp4` avec vidéo + audio générés par IA.

---

### Prérequis

1. **Google Colab Pro** (~10$/mois) — [colab.research.google.com](https://colab.research.google.com)
2. Sélectionner **Runtime > Change runtime type > A100 GPU**
3. Un **token HuggingFace** (gratuit) — [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
4. Accepter la licence **Gemma 3** — [huggingface.co/google/gemma-3-1b-pt](https://huggingface.co/google/gemma-3-1b-pt)

## Étape 1 — Vérifier le GPU

Exécutez cette cellule pour confirmer que vous avez bien un **A100**.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "❌ Pas de GPU détecté !\n"
        "   → Allez dans Runtime > Change runtime type > GPU (A100)\n"
        "   → Si A100 n'apparaît pas, vérifiez votre abonnement Colab Pro"
    )

gpu_name = torch.cuda.get_device_name(0)
vram_total = torch.cuda.get_device_properties(0).total_memory / 1024**3
vram_free = torch.cuda.mem_get_info(0)[0] / 1024**3

print(f"✅ GPU détecté : {gpu_name}")
print(f"   VRAM : {vram_total:.0f} GB total, {vram_free:.0f} GB libre")
print(f"   PyTorch : {torch.__version__}")
print(f"   CUDA : {torch.version.cuda}")

if vram_total < 20:
    print()
    print("⚠️  ATTENTION : Vous avez un GPU avec < 20 GB VRAM.")
    print("   Le modèle 19B FP8 nécessite ~20 GB minimum.")
    print("   → Changez pour un A100 : Runtime > Change runtime type > A100")
elif vram_total >= 35:
    print()
    print("🎉 Parfait ! A100 40GB détecté — idéal pour AIPROD.")
else:
    print()
    print("✅ GPU suffisant pour la génération vidéo.")

## Étape 2 — Cloner le repo AIPROD

Clone le code source (~7 MB). Les modèles seront téléchargés séparément.

In [ ]:
import os

REPO_URL = "https://github.com/Blockprod/AIPROD.git"
WORK_DIR = "/content/AIPROD"

# Si repo privé, décommentez et ajoutez votre token GitHub :
# REPO_URL = "https://<VOTRE_TOKEN_GITHUB>@github.com/Blockprod/AIPROD.git"

if not os.path.exists(WORK_DIR):
    !git clone {REPO_URL} {WORK_DIR}
else:
    !cd {WORK_DIR} && git pull origin main

%cd {WORK_DIR}
print(f"\n✅ Repo AIPROD prêt dans {os.getcwd()}")

## Étape 3 — Installer les dépendances

Installe PyTorch, **diffusers** (pipeline officiel Lightricks LTX-2) et les utilitaires (~2-3 min).

In [ ]:
# Installer PyTorch avec CUDA 12.1
%pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Diffusers (dernière version git pour support LTX-2)
%pip install -q git+https://github.com/huggingface/diffusers

# Dépendances essentielles
%pip install -q transformers accelerate safetensors huggingface_hub
%pip install -q sentencepiece protobuf  # Pour le tokenizer Gemma 3
%pip install -q pillow imageio rich pyyaml tqdm
%pip install -q av  # PyAV pour l'encodage vidéo MP4

print()
print("✅ Installation terminée")
print(f"   torch : {__import__('torch').__version__}")
print(f"   diffusers : {__import__('diffusers').__version__}")
print(f"   transformers : {__import__('transformers').__version__}")

## Étape 4 — Authentification HuggingFace

Requis pour télécharger le text encoder **Gemma 3** (modèle gated par Google).

> **Comment obtenir un token :**
> 1. Créez un compte sur [huggingface.co](https://huggingface.co)
> 2. Acceptez la licence Gemma : [google/gemma-3-1b-pt](https://huggingface.co/google/gemma-3-1b-pt)
> 3. Créez un token : [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) (type **Read**)
> 4. Dans Colab : panneau gauche 🔑 > **Add secret** > Name: `HF_TOKEN`, Value: votre token

In [ ]:
from huggingface_hub import login

# ═══════════════════════════════════════════════════════════════
# Authentification HuggingFace (requis pour Gemma 3)
# ═══════════════════════════════════════════════════════════════
try:
    from google.colab import userdata  # type: ignore[import-not-found]
    hf_token = userdata.get("HF_TOKEN")
    login(token=hf_token)
    print("✅ Authentifié sur HuggingFace via Colab Secret")
except Exception:
    hf_token = None
    print("⚠️  Pas de HF_TOKEN trouvé dans les Secrets Colab.")
    print()
    print("   Pour obtenir un token :")
    print("   1. Allez sur https://huggingface.co/google/gemma-3-1b-pt → Accepter la licence")
    print("   2. Créez un token : https://huggingface.co/settings/tokens")
    print("   3. Panneau gauche 🔑 > Add secret > Name: HF_TOKEN")
    print("   4. Relancez cette cellule")
    raise PermissionError("HF_TOKEN requis pour télécharger Gemma 3")

## Étape 5 — GÉNÉRER VOTRE VIDÉO ! 🎬

**Modifiez le `PROMPT` ci-dessous**, puis exécutez la cellule.

| Paramètre | Valeur par défaut | Description |
|-----------|-------------------|-------------|
| `PROMPT` | *Un drone survolant...* | Votre description de la vidéo |
| `NUM_FRAMES` | 121 | Nombre d'images (~5 sec à 24fps) |
| `STEPS` | 40 | Qualité (plus = meilleur mais plus lent) |
| `HEIGHT × WIDTH` | 512 × 768 | Résolution |
| `GUIDANCE_SCALE` | 4.0 | Fidélité au prompt (3-7) |

> **Premier lancement :** le modèle LTX-2 (~20 GB) sera téléchargé automatiquement.  
> **Durée estimée :** 10-15 min (download) + 3-5 min (génération)  
> Les runs suivants dans la même session utilisent le cache.

In [ ]:
# ╔═══════════════════════════════════════════════════════════════╗
# ║  CONFIGUREZ VOTRE VIDÉO ICI                                 ║
# ╚═══════════════════════════════════════════════════════════════╝

PROMPT = "A cinematic slow-motion drone shot over misty mountains at golden hour, \
volumetric god rays piercing through dramatic clouds, \
camera slowly pulling back to reveal a vast valley below, \
professional cinematography, 4K, shallow depth of field"

NEGATIVE_PROMPT = "worst quality, inconsistent motion, blurry, jittery, distorted, \
shaky, glitchy, low quality, deformed, motion smear, motion artifacts, \
fused fingers, bad anatomy, weird hand, ugly, transition, static"

# Paramètres de génération
NUM_FRAMES = 121      # 121 = ~5 sec à 24fps (doit être divisible par 8 + 1)
STEPS = 40            # 30 = rapide, 40 = standard, 50 = haute qualité
HEIGHT = 512          # Résolution verticale (divisible par 32)
WIDTH = 768           # Résolution horizontale (divisible par 32)
GUIDANCE_SCALE = 4.0  # 3.0-7.0 recommandé (fidélité au prompt)
SEED = 42             # Changez pour un résultat différent

# ╔═══════════════════════════════════════════════════════════════╗
# ║  NE PAS MODIFIER EN-DESSOUS (sauf si vous savez ce que      ║
# ║  vous faites)                                                ║
# ╚═══════════════════════════════════════════════════════════════╝

import os
import re
import time
import torch
from diffusers import LTX2Pipeline
from diffusers.pipelines.ltx2.export_utils import encode_video

# Créer dossier de sortie
OUTPUT_DIR = "/content/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Nom de fichier basé sur le prompt
safe_name = re.sub(r"[^a-zA-Z0-9_ ]", "", PROMPT)[:50].strip().replace(" ", "_")
OUTPUT_PATH = f"{OUTPUT_DIR}/{safe_name}.mp4"

print("═" * 60)
print("🎬 AIPROD × LTX-2 — Génération Vidéo + Audio")
print("═" * 60)
print(f"   Prompt   : {PROMPT[:80]}..." if len(PROMPT) > 80 else f"   Prompt   : {PROMPT}")
print(f"   Résolution : {HEIGHT}×{WIDTH}")
print(f"   Frames   : {NUM_FRAMES} (~{NUM_FRAMES/24:.1f}s à 24fps)")
print(f"   Steps    : {STEPS}")
print(f"   Guidance : {GUIDANCE_SCALE}")
print(f"   Seed     : {SEED}")
print(f"   Output   : {OUTPUT_PATH}")
print()

# ── Charger le Pipeline LTX-2 ─────────────────────────────────
print("⏳ Chargement du pipeline LTX-2 19B...")
print("   (Premier lancement : téléchargement ~20 GB depuis HuggingFace)")
t_load = time.time()

pipe = LTX2Pipeline.from_pretrained(
    "Lightricks/LTX-2",
    torch_dtype=torch.bfloat16,
)
# Offload séquentiel : seul le composant actif est sur GPU
# Permet de faire tourner le modèle 19B sur A100 40GB
pipe.enable_sequential_cpu_offload(device="cuda:0")
pipe.vae.enable_tiling()

print(f"✅ Pipeline chargé en {time.time() - t_load:.0f}s")
print()

# ── Générer ────────────────────────────────────────────────────
print(f"🎨 Génération en cours ({STEPS} étapes de diffusion)...")
t_gen = time.time()

generator = torch.Generator("cpu").manual_seed(SEED)

video, audio = pipe(
    prompt=PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    width=WIDTH,
    height=HEIGHT,
    num_frames=NUM_FRAMES,
    frame_rate=24.0,
    num_inference_steps=STEPS,
    guidance_scale=GUIDANCE_SCALE,
    generator=generator,
    output_type="np",
    return_dict=False,
)

gen_time = time.time() - t_gen
print(f"✅ Génération terminée en {gen_time:.0f}s")
print()

# ── Sauvegarder en MP4 ────────────────────────────────────────
print("💾 Encodage MP4 (vidéo + audio)...")

encode_video(
    video[0],
    fps=24.0,
    audio=audio[0].float().cpu() if audio is not None else None,
    audio_sample_rate=pipe.vocoder.config.output_sampling_rate,
    output_path=OUTPUT_PATH,
)

# Libérer la VRAM
del pipe, video, audio, generator
torch.cuda.empty_cache()

# ── Résultat ──────────────────────────────────────────────────
from pathlib import Path
mp4 = Path(OUTPUT_PATH)
size_mb = mp4.stat().st_size / 1024**2
duration = NUM_FRAMES / 24.0

print()
print("═" * 60)
print("🎉 VIDÉO GÉNÉRÉE AVEC SUCCÈS !")
print("═" * 60)
print(f"   📁 Fichier  : {OUTPUT_PATH}")
print(f"   💾 Taille   : {size_mb:.1f} MB")
print(f"   ⏱️  Durée    : {duration:.1f} secondes")
print(f"   📐 Résolution : {HEIGHT}×{WIDTH}")
print(f"   🔊 Audio    : {'Oui' if mp4.stat().st_size > 1024 else 'Non'}")
print(f"   🕐 Temps    : {gen_time:.0f}s")
print()
print("   Exécutez la cellule suivante pour télécharger le fichier.")

## Étape 6 — Télécharger votre vidéo

**Option A :** Téléchargement direct dans votre navigateur  
**Option B :** Sauvegarder sur Google Drive

In [ ]:
# ═══════════════════════════════════════════════════════════════
# OPTION A : Téléchargement direct (navigateur)
# ═══════════════════════════════════════════════════════════════
from google.colab import files  # type: ignore[import-not-found]
files.download(OUTPUT_PATH)
print(f"📥 Téléchargement lancé : {OUTPUT_PATH}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# OPTION B : Sauvegarder sur Google Drive
# ═══════════════════════════════════════════════════════════════
from google.colab import drive  # type: ignore[import-not-found]
import shutil

drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/AIPROD/videos'
os.makedirs(DRIVE_DIR, exist_ok=True)

dst = f"{DRIVE_DIR}/{os.path.basename(OUTPUT_PATH)}"
shutil.copy2(OUTPUT_PATH, dst)
print(f"✅ Vidéo sauvegardée sur Drive : {dst}")

## Étape 7 — Visualiser la vidéo dans Colab

Affiche la vidéo directement dans le notebook.

In [ ]:
from IPython.display import HTML
from base64 import b64encode

with open(OUTPUT_PATH, "rb") as f:
    video_data = f.read()

b64 = b64encode(video_data).decode("ascii")
HTML(f"""
<h3>🎬 Résultat : {os.path.basename(OUTPUT_PATH)}</h3>
<video width="768" controls autoplay loop>
  <source src="data:video/mp4;base64,{b64}" type="video/mp4">
</video>
<p><b>Prompt :</b> {PROMPT}</p>
""")

---

## Bonus — Générer plusieurs vidéos

Modifiez les prompts et lancez cette cellule pour générer un lot de vidéos.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# GÉNÉRATION EN LOT — Modifiez les prompts ci-dessous
# ═══════════════════════════════════════════════════════════════

BATCH_PROMPTS = [
    "A peaceful zen garden with cherry blossoms falling in slow motion",
    "A futuristic city at night with neon lights and flying cars",
    "Ocean waves crashing on a tropical beach at sunset, cinematic",
]

BATCH_STEPS = 40
BATCH_FRAMES = 121
BATCH_HEIGHT = 512
BATCH_WIDTH = 768
BATCH_GUIDANCE = 4.0

# ═══════════════════════════════════════════════════════════════

import os, re, time, torch
from diffusers import LTX2Pipeline
from diffusers.pipelines.ltx2.export_utils import encode_video

NEGATIVE = "worst quality, inconsistent motion, blurry, jittery, distorted"

# Charger le pipeline une seule fois
print("⏳ Chargement du pipeline...")
pipe = LTX2Pipeline.from_pretrained(
    "Lightricks/LTX-2",
    torch_dtype=torch.bfloat16,
)
pipe.enable_sequential_cpu_offload(device="cuda:0")
pipe.vae.enable_tiling()
print("✅ Pipeline chargé\n")

for i, prompt in enumerate(BATCH_PROMPTS, 1):
    safe_name = re.sub(r"[^a-zA-Z0-9_ ]", "", prompt)[:50].strip().replace(" ", "_")
    out_path = f"/content/output/{safe_name}.mp4"
    
    print(f"\n{'═'*60}")
    print(f"🎬 Vidéo {i}/{len(BATCH_PROMPTS)} : {prompt[:60]}")
    print(f"{'═'*60}")
    
    t0 = time.time()
    generator = torch.Generator("cpu").manual_seed(42 + i)
    
    video, audio = pipe(
        prompt=prompt,
        negative_prompt=NEGATIVE,
        width=BATCH_WIDTH,
        height=BATCH_HEIGHT,
        num_frames=BATCH_FRAMES,
        frame_rate=24.0,
        num_inference_steps=BATCH_STEPS,
        guidance_scale=BATCH_GUIDANCE,
        generator=generator,
        output_type="np",
        return_dict=False,
    )
    
    encode_video(
        video[0],
        fps=24.0,
        audio=audio[0].float().cpu() if audio is not None else None,
        audio_sample_rate=pipe.vocoder.config.output_sampling_rate,
        output_path=out_path,
    )
    
    size_mb = os.path.getsize(out_path) / 1024**2
    elapsed = time.time() - t0
    print(f"✅ Sauvegardé : {out_path} ({size_mb:.1f} MB, {elapsed:.0f}s)")
    
    del video, audio
    torch.cuda.empty_cache()

# Libérer le pipeline
del pipe
torch.cuda.empty_cache()

print(f"\n{'═'*60}")
print(f"🎉 {len(BATCH_PROMPTS)} vidéos générées dans /content/output/")
print(f"{'═'*60}")

# Lister les fichiers
for f in sorted(os.listdir("/content/output")):
    if f.endswith(".mp4"):
        size = os.path.getsize(f"/content/output/{f}") / 1024**2
        print(f"   📹 {f} ({size:.1f} MB)")

---

## Bonus — Génération avec LoRA AIPROD (après entraînement)

Si vous avez entraîné le modèle LoRA SHDT (notebook d'entraînement),  
vous pouvez l'appliquer ici pour des résultats personnalisés.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# GÉNÉRATION AVEC LORA (optionnel — après entraînement)
# ═══════════════════════════════════════════════════════════════
# Décommentez et adaptez le chemin vers votre fichier LoRA :
#
# import torch
# from diffusers import LTX2Pipeline
#
# pipe = LTX2Pipeline.from_pretrained("Lightricks/LTX-2", torch_dtype=torch.bfloat16)
# pipe.enable_sequential_cpu_offload(device="cuda:0")
#
# # Charger votre LoRA entraîné
# LORA_PATH = "/content/drive/MyDrive/AIPROD/trained_models/my-lora.safetensors"
# pipe.load_lora_weights(LORA_PATH, adapter_name="my_style")
# pipe.set_adapters("my_style", 0.8)  # 0.0 = pas d'effet, 1.0 = effet maximum
#
# video, audio = pipe(
#     prompt="Your prompt here",
#     negative_prompt="worst quality, blurry, distorted",
#     width=768, height=512,
#     num_frames=121,
#     num_inference_steps=40,
#     guidance_scale=4.0,
#     output_type="np",
#     return_dict=False,
# )

print("ℹ️ Décommentez le code ci-dessus après avoir entraîné votre LoRA.")
print("   Le modèle LTX-2 supporte nativement les LoRA via diffusers.")

---

## FAQ & Dépannage

### ❌ `CUDA out of memory`
- Assurez-vous d'avoir un **A100** (pas T4 ni V100)
- Réduisez `NUM_FRAMES` à 61 ou `HEIGHT/WIDTH` à 384×512
- Le pipeline utilise `enable_sequential_cpu_offload` pour limiter la VRAM

### ❌ `401 Client Error: Unauthorized`
- Vérifiez votre `HF_TOKEN` dans les Secrets Colab (🔑)
- Acceptez la licence Gemma 3 : [huggingface.co/google/gemma-3-1b-pt](https://huggingface.co/google/gemma-3-1b-pt)

### ❌ `No module named 'diffusers.pipelines.ltx2'`
- Relancez la cellule 3 — diffusers doit être installé depuis git
- Vérifiez que la commande `pip install git+...` s'est terminée sans erreur

### ❌ Vidéo toute noire / bruit
- Essayez un `SEED` différent (ex: 42, 123, 7)
- Augmentez `STEPS` à 50
- Augmentez `GUIDANCE_SCALE` à 5.0 ou 6.0
- Utilisez un prompt plus descriptif en anglais

### 💡 Astuces pour de bons prompts
- Utilisez l'**anglais** pour de meilleurs résultats
- Soyez **descriptif** : mouvement de caméra, éclairage, style
- Ajoutez des mots clés : `cinematic`, `4K`, `slow motion`, `aerial shot`
- Exemples :
  - `"A slow-motion close-up of rain drops falling on a flower petal"`
  - `"Aerial shot of a mountain lake at sunrise, mist rising, 4K cinematic"`
  - `"A cat sitting on a windowsill watching snowfall, cozy room, warm light"`

### 🔊 Audio
- LTX-2 génère **vidéo + audio synchronisés** en un seul modèle
- L'audio est automatiquement inclus dans le MP4
- Pour des résultats audio optimaux, décrivez l'ambiance sonore dans le prompt

### 💰 Coût estimé
- Colab Pro : ~10$/mois = **illimité** (dans les limites GPU)
- ~5 min par vidéo → **~50-100 vidéos par session**
- Pas de frais supplémentaires par vidéo